# Resync + AAS Gradient Artifact Removal Demo

Este notebook carga un registro simultaneo EEG-fMRI (`fmrirestingec`), aplica primero la resincronizacion global `Resync` para reajustar la senal EEG al ritmo del fMRI y, despues, ejecuta el pipeline propio de `AAS` definido en `aas_Resync.py`. Incluye visualizacion interactiva completa de los canales EEG con `mne` antes de la limpieza, despues de `Resync` y despues de `Resync + AAS`.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import mne
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from functions.aas_Resync import run_resync_aas_pipeline

In [2]:
DEFAULT_EEG_ROOT = Path(
    r"C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG"
)
SUBJECT = "sub-007"
TASK = "fmrirestingec"
TR = 2.0
REFERENCE_CHANNEL = 9
EXCLUDE_LAST_CHANNEL = True
SEARCH_BOUNDS = (0.995, 1.005)
AAS_WINDOW_SIZE = 21
DISPLAY_SECONDS = 60
DISPLAY_SEGMENT_INDEX = 0

In [3]:
def get_eeg_set_path(subject: str, task: str = TASK, eeg_root: Path = DEFAULT_EEG_ROOT) -> Path:
    eeg_path = eeg_root / subject / "eeg" / f"{subject}_task-{task}_eeg.set"
    if not eeg_path.exists():
        raise FileNotFoundError(f"EEG file not found: {eeg_path}")
    return eeg_path


def load_raw_eeg(eeg_path: Path) -> mne.io.BaseRaw:
    return mne.io.read_raw_eeglab(eeg_path, preload=True, verbose="ERROR")


def prepare_raw_for_pipeline(raw: mne.io.BaseRaw, exclude_last_channel: bool = True) -> mne.io.BaseRaw:
    if exclude_last_channel:
        return raw.copy().pick(raw.ch_names[:-1])
    return raw.copy()


def make_raw_from_array(data: np.ndarray, raw_reference: mne.io.BaseRaw) -> mne.io.BaseRaw:
    info = raw_reference.info.copy()
    return mne.io.RawArray(data.copy(), info, verbose="ERROR")


def _windowed_signal(data: np.ndarray, duration_s: float, fs: float) -> tuple[np.ndarray, np.ndarray]:
    n_samples = min(int(round(duration_s * fs)), data.shape[-1])
    time = np.arange(n_samples) / fs
    return time, data[..., :n_samples]


def plot_before_after(
    raw_before: mne.io.BaseRaw,
    resynced_data: np.ndarray,
    cleaned_data: np.ndarray,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    before = raw_before.get_data(picks=[channel_index])[0]
    resynced = resynced_data[channel_index]
    cleaned = cleaned_data[channel_index]

    time_before, before = _windowed_signal(before[np.newaxis, :], duration_s, fs)
    time_resynced, resynced = _windowed_signal(resynced[np.newaxis, :], duration_s, fs)
    time_cleaned, cleaned = _windowed_signal(cleaned[np.newaxis, :], duration_s, fs)
    channel_name = raw_before.ch_names[channel_index]

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=False)
    axes[0].plot(time_before, before[0], linewidth=0.8)
    axes[0].set_title(f"Raw EEG before Resync/AAS | {channel_name}")
    axes[0].set_ylabel("Amplitude")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(time_resynced, resynced[0], linewidth=0.8, color="tab:green")
    axes[1].set_title(f"After Resync | {channel_name}")
    axes[1].set_ylabel("Amplitude")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(time_cleaned, cleaned[0], linewidth=0.8, color="tab:orange")
    axes[2].set_title(f"After Resync + AAS | {channel_name}")
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Amplitude")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_overlay_before_after(
    raw_before: mne.io.BaseRaw,
    cleaned_data: np.ndarray,
    channel_index: int = 0,
    duration_s: float = DISPLAY_SECONDS,
) -> None:
    fs = float(raw_before.info["sfreq"])
    before = raw_before.get_data(picks=[channel_index])[0]
    cleaned = cleaned_data[channel_index]

    time_before, before = _windowed_signal(before[np.newaxis, :], duration_s, fs)
    time_cleaned, cleaned = _windowed_signal(cleaned[np.newaxis, :], duration_s, fs)
    channel_name = raw_before.ch_names[channel_index]

    plt.figure(figsize=(14, 4))
    plt.plot(time_before, before[0], label="Before cleaning", linewidth=0.8, alpha=0.75)
    plt.plot(time_cleaned, cleaned[0], label="After Resync + AAS", linewidth=0.8, alpha=0.85)
    plt.title(f"Overlay Before/After | {channel_name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_reference_segment_comparison(result: dict, channel_name: str, segment_index: int = DISPLAY_SEGMENT_INDEX) -> None:
    raw_segment = result["aas"]["segmented_signal"][REFERENCE_CHANNEL, segment_index]
    cleaned_segment = result["aas"]["cleaned_segments"][REFERENCE_CHANNEL, segment_index]
    local_template = result["aas"]["local_templates"][REFERENCE_CHANNEL, segment_index]
    reference_template = result["aas"]["reference_template"]
    samples = np.arange(reference_template.shape[0])

    plt.figure(figsize=(14, 5))
    plt.plot(samples, raw_segment, label="Segment after Resync", linewidth=0.8, alpha=0.8)
    plt.plot(samples, local_template, label="Local AAS template", linewidth=1.0, alpha=0.9)
    plt.plot(samples, reference_template, label="Reference template", linewidth=1.2)
    plt.plot(samples, cleaned_segment, label="Cleaned segment", linewidth=0.8, alpha=0.8)
    plt.title(f"Reference TR Segment Comparison | {channel_name}")
    plt.xlabel("Samples within TR")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_resync_lags(result: dict) -> None:
    lags = np.asarray(result["resync"]["lags"], dtype=float)
    coherence = np.asarray(result["resync"]["coherence_scores"], dtype=float)
    segments = np.arange(lags.size)

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(segments, lags, marker="o", markersize=3, linewidth=0.8)
    axes[0].set_title("Resync estimated lag per TR segment")
    axes[0].set_ylabel("Lag (samples)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(segments, coherence, marker="o", markersize=3, linewidth=0.8, color="tab:red")
    axes[1].set_title("Resync normalized coherence per TR segment")
    axes[1].set_xlabel("Segment index")
    axes[1].set_ylabel("Coherence")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [4]:
eeg_path = get_eeg_set_path(SUBJECT)
raw = load_raw_eeg(eeg_path)
raw_eeg = prepare_raw_for_pipeline(raw, exclude_last_channel=EXCLUDE_LAST_CHANNEL)
eeg_data = raw_eeg.get_data()
fs = float(raw_eeg.info["sfreq"])

print(f"Subject: {SUBJECT}")
print(f"Task: {TASK}")
print(f"EEG path: {eeg_path}")
print(f"Original shape: {raw.get_data().shape}")
print(f"Pipeline input shape: {eeg_data.shape}")
print(f"Sampling frequency: {fs} Hz")
print(f"Reference channel: {REFERENCE_CHANNEL} ({raw_eeg.ch_names[REFERENCE_CHANNEL]})")
if EXCLUDE_LAST_CHANNEL:
    print(f"Excluded channel: {raw.ch_names[-1]}")

Subject: sub-007
Task: fmrirestingec
EEG path: C:\Users\pedro\Documents\DOCTORADO_Pedro\Code\phD_Repository\data\raw\Dataset1\Simultaneous_EEG_fMRI\BIDS_dataset_EEG\sub-007\eeg\sub-007_task-fmrirestingec_eeg.set
Original shape: (33, 628889)
Pipeline input shape: (32, 628889)
Sampling frequency: 1000.0 Hz
Reference channel: 9 (E10)
Excluded channel: ECG


## Visualizacion interactiva del EEG original

In [5]:
mne.viz.set_browser_backend("qt")
raw_eeg.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

Using qt as 2D backend.
Channels marked as bad:
none


## Ejecucion del pipeline Resync + AAS

In [6]:
result = run_resync_aas_pipeline(
    eeg_data,
    TR=TR,
    fs=fs,
    reference_channel=REFERENCE_CHANNEL,
)

cleaned_eeg = result["cleaned_signal"]
resynced_eeg = result["resynced_signal"]
print(f"Start fMRI: {result["fmri_start_time_sec"]}")
print(f"Finish fMRI: {result["fmri_end_time_sec"]}")
print(f"Fs_resynced: {result["fs_resynced"]}")
print(f"n_resynced_samples: {result["n_resynced_samples"]}")

Start fMRI: 5.454
Finish fMRI: 626.584
Fs_resynced: 1000.0
n_resynced_samples: 628889


## Visualizacion interactiva despues de Resync

In [7]:
raw_clean = raw_eeg.copy()
raw_clean._data = cleaned_eeg.copy()

In [8]:
raw_clean.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)

Channels marked as bad:
none


In [ ]:
raw_resynced = raw_eeg.copy()
raw_resynced._data = resynced_eeg.copy()

In [ ]:
raw_resynced.plot(
    n_channels=20,
    duration=10,
    show_scrollbars=True,
    block=True,
)